In [1]:
import pandas as pd
import numpy as np

id_cols = {"ZIPCODE": "string", "BBL": "string"}
insp = pd.read_csv("data/inspection_level.csv", dtype=id_cols,
                   parse_dates=["INSPECTION DATE"], low_memory=False)
print("Inspections:", insp.shape)

keep = ["BBL", "yearbuilt", "bldgclass", "landuse", "numfloors",
        "unitsres", "unitstotal", "bldgarea", "lotarea", "assesstot"]
pluto = pd.read_csv("data/external/pluto.csv", dtype=str, usecols=keep, low_memory=False)

pluto["BBL"] = pluto["BBL"].str.replace(r"\.0$", "", regex=True).str.zfill(10)
for c in ["yearbuilt", "numfloors", "unitsres", "unitstotal", "bldgarea", "lotarea", "assesstot"]:
    pluto[c] = pd.to_numeric(pluto[c], errors="coerce")
pluto.loc[pluto["yearbuilt"] == 0, "yearbuilt"] = np.nan
pluto = pluto.drop_duplicates("BBL")
print("PLUTO buildings:", pluto.shape)

insp["BBL"] = insp["BBL"].str.replace(r"\.0$", "", regex=True).str.zfill(10)
df = insp.merge(pluto, on="BBL", how="left")

print("Rows after join:", len(df), "(must be 93,292)")
print("Matched to a building:", round(df["yearbuilt"].notna().mean() * 100, 1), "%")

Inspections: (93292, 43)
PLUTO buildings: (858312, 10)
Rows after join: 93292 (must be 93,292)
Matched to a building: 96.4 %


In [2]:
df["pluto_matched"] = df["BBL"].isin(set(pluto["BBL"])).astype(int)

print("Inspections with a BBL:", df["BBL"].notna().sum())
print("Matched to PLUTO:", df["pluto_matched"].sum(),
      "(", round(df["pluto_matched"].mean() * 100, 1), "% of all inspections )")
print("Year built known:", df["yearbuilt"].notna().sum(),
      "(", round(df["yearbuilt"].notna().mean() * 100, 1), "% )")
print("Building area known:", df["bldgarea"].notna().sum())

Inspections with a BBL: 91209
Matched to PLUTO: 90473 ( 97.0 % of all inspections )
Year built known: 89978 ( 96.4 % )
Building area known: 90461


In [3]:
import requests

key = open("data/external/census_key.txt").read().strip()
zips = sorted(df["ZIPCODE"].dropna().str[:5].unique())
print("ZIP codes in our data:", len(zips))

acs = None
for year in [2024, 2023]:
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": "NAME,B19013_001E,B01003_001E",
              "for": "zip code tabulation area:*", "key": key}
    r = requests.get(url, params=params, timeout=60)
    print(year, "status:", r.status_code)
    if r.status_code == 200:
        rows = r.json()
        acs = pd.DataFrame(rows[1:], columns=rows[0])
        acs_year = year
        break
    else:
        print(r.text[:200].replace(key, "***"))

if acs is not None:
    acs = acs.rename(columns={"B19013_001E": "median_hh_income",
                              "B01003_001E": "population",
                              "zip code tabulation area": "ZIPCODE"})
    for c in ["median_hh_income", "population"]:
        acs[c] = pd.to_numeric(acs[c], errors="coerce")
        acs.loc[acs[c] < 0, c] = np.nan
    acs = acs[["ZIPCODE", "median_hh_income", "population"]]
    acs.to_csv("data/external/acs_zip.csv", index=False)
    print("ACS", acs_year, "5-year, ZIP areas:", acs.shape)
    print(acs[acs["ZIPCODE"].isin(zips)].describe().round(0))

ZIP codes in our data: 238
2024 status: 200
ACS 2024 5-year, ZIP areas: (33772, 3)
       median_hh_income  population
count             196.0       218.0
mean            99620.0     41119.0
std             46675.0     28070.0
min             24086.0         0.0
25%             69222.0     20672.0
50%             89988.0     38569.0
75%            117385.0     62434.0
max            250001.0    105716.0


In [4]:
df["ZIP5"] = df["ZIPCODE"].str[:5]
acs2 = acs.rename(columns={"ZIPCODE": "ZIP5"})
n_before = len(df)
df = df.merge(acs2, on="ZIP5", how="left")

print("Rows after Census join:", len(df), "(must be", n_before, ")")
print("ZIP matched in ACS:", round(df["population"].notna().mean() * 100, 1), "%")
print("Median income known:", round(df["median_hh_income"].notna().mean() * 100, 1), "%")

df.to_csv("data/inspection_level_plus.csv", index=False)
print("Saved:", df.shape)

Rows after Census join: 93292 (must be 93292 )
ZIP matched in ACS: 98.5 %
Median income known: 97.6 %
Saved: (93292, 56)
